In [1]:
%pip install rdflib owlready2 -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install  graphviz pydotplus -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!apt-get install -y openjdk-11-jdk   -qq

"apt-get" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


In [4]:
import pandas as pd
import os
from owlready2 import *

## Загрузка данных

In [5]:
# Загружаем данные из CSV файлов
def load_data():
    knowledge_df = pd.read_csv('C:/Users/User/big_data_course_2026/task_8_Ontology/ontology_data/ontology_data/knowledge.csv')
    competencies_df = pd.read_csv('C:/Users/User/big_data_course_2026/task_8_Ontology/ontology_data/ontology_data/competencies.csv')
    competency_knowledge_df = pd.read_csv('C:/Users/User/big_data_course_2026/task_8_Ontology/ontology_data/ontology_data/competency_knowledge.csv')
    positions_df = pd.read_csv('C:/Users/User/big_data_course_2026/task_8_Ontology/ontology_data/ontology_data/positions.csv')
    position_competency_df = pd.read_csv('C:/Users/User/big_data_course_2026/task_8_Ontology/ontology_data/ontology_data/position_competency.csv')
    employees_df = pd.read_csv('C:/Users/User/big_data_course_2026/task_8_Ontology/ontology_data/ontology_data/employees.csv')

    return knowledge_df, competencies_df, competency_knowledge_df, positions_df, position_competency_df, employees_df

In [6]:
knowledge_df, competencies_df, competency_knowledge_df, positions_df, position_competency_df, employees_df = load_data()

In [7]:
knowledge_df

,knowledge_id,knowledge_name
0,1,Математическая статистика
1,2,Линейная алгебра
2,3,Теория вероятностей
3,4,Python программирование
4,5,"Библиотеки ML (scikit-learn, TensorFlow, PyTorch)"
5,6,Обработка естественного языка (NLP)
6,7,Компьютерное зрение
7,8,"Работа с большими данными (Spark, Hadoop)"
8,9,SQL и базы данных
9,10,Визуализация данных


# Создание новой онтологии

## Итерация 1.
Создадим знания и компетенции

In [8]:
# Создаем пустую онтологию с уникальным идентификатором
onto = get_ontology("http://test.org/ml_specialist_v1.owl")

with onto:  # Контекстный менеджер - все изменения внутри него сохранятся в онтологию
    
    # Определяем КЛАССЫ (типы сущностей)
    # Thing - базовый класс в OWL, от которого наследуются все классы
    class Знание(Thing):
        """Класс для представления конкретных знаний (Python, SQL, математика)"""
        pass
    
    class Компетенция(Thing):
        """Класс для представления компетенций (наборов знаний)"""
        pass
    
    # Определяем СВОЙСТВА (отношения между классами)
    # ObjectProperty - связывает два объекта (класса)
    class требует_знание(ObjectProperty):
        """Свойство: компетенция требует знание"""
        domain = [Компетенция]  # От кого (субъект)
        range = [Знание]        # К кому (объект)
    
    # DataProperty - связывает объект с конкретным значением (строкой, числом)
    class название(DataProperty):
        """Свойство для хранения текстового названия"""
        domain = [Thing]        # Любой класс может иметь название
        range = [str]           # Значение - строка

### Создание индивидуальностей для знаний и компетенций



In [9]:
def create_knowledge(knowledge_df):
  with onto:

    knowledge_instances = {}
    for _, row in knowledge_df.iterrows():
        knowledge = Знание(f"знание_{row['knowledge_id']}") # Создаем экземпляр класса Знание с уникальным именем
        knowledge.название = [row['knowledge_name']] # Добавляем свойство "название"
        knowledge_instances[row['knowledge_id']] = knowledge
    return knowledge_instances

def create_competencies( competencies_df):
  with onto:
    competency_instances = {}
    for _, row in competencies_df.iterrows():
        competency = Компетенция(f"компетенция_{row['competency_id']}")
        competency.название = [row['competency_name']]
        competency_instances[row['competency_id']] = competency
    return competency_instances


In [10]:
knowledges = create_knowledge(knowledge_df)

In [11]:
knowledges

{1: ml_specialist_v1.знание_1,
 2: ml_specialist_v1.знание_2,
 3: ml_specialist_v1.знание_3,
 4: ml_specialist_v1.знание_4,
 5: ml_specialist_v1.знание_5,
 6: ml_specialist_v1.знание_6,
 7: ml_specialist_v1.знание_7,
 8: ml_specialist_v1.знание_8,
 9: ml_specialist_v1.знание_9,
 10: ml_specialist_v1.знание_10}

In [12]:
for id, item in knowledges.items():
  print (item.название[0])

Математическая статистика
Линейная алгебра
Теория вероятностей
Python программирование
Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
Обработка естественного языка (NLP)
Компьютерное зрение
Работа с большими данными (Spark, Hadoop)
SQL и базы данных
Визуализация данных


In [13]:
competencies = create_competencies( competencies_df)
for id, item in competencies.items():
  print (item.название[0])

Математическая подготовка
Программирование и алгоритмы
Машинное обучение
Глубокое обучение
Обработка и анализ данных
Работа с большими данными
Визуализация и представление данных
Развертывание ML-моделей


### Создание отношений между знаниями и компетенциями

In [14]:
# Связываем компетенции со знаниями
def  relate_competencies_knowledge (competency_knowledge_df,  competency_instances, knowledge_instances):
  with onto:
    for _, row in competency_knowledge_df.iterrows():
        competency = competency_instances[row['competency_id']] # Извлекаем объект компетенции по ID из словаря
        knowledge = knowledge_instances[row['knowledge_id']] # Извлекаем объект знания по ID из словаря
        competency.требует_знание.append(knowledge)# Добавляем связь: competency требует knowledge, .append() - добавляем знание в список требуемых знаний


In [15]:
relate_competencies_knowledge (competency_knowledge_df, competencies, knowledges)

Выведем связи между компенциями и знаниями

In [16]:
def print_competency_knowledge():
  print("\n" + "="*60)
  print("СВЯЗИ ЗНАНИЙ И КОМПЕТЕНЦИЙ")
  print("="*60)

  for competency_id, competency in competencies.items(): # items() - возвращает пары (ключ, значение) из словаря
      comp_name = competency.название[0] # Берем название компетенции (первое значение свойства)
      print(f"\n{comp_name} требует знания:")

      required_knowledge = list(competency.требует_знание) # list() - преобразуем итератор в список
      for know in required_knowledge:
          print(f"  - {know.название[0]}")

print_competency_knowledge()



СВЯЗИ ЗНАНИЙ И КОМПЕТЕНЦИЙ

Математическая подготовка требует знания:
  - Математическая статистика
  - Линейная алгебра
  - Теория вероятностей

Программирование и алгоритмы требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Машинное обучение требует знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Глубокое обучение требует знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Обработка и анализ данных требует знания:
  - Математическая статистика
  - SQL и базы данных

Работа с большими данными требует знания:
  - Работа с большими данными (Spark, Hadoop)
  - SQL и базы данных

Визуализация и представление данных требует знания:
  - Визуализация данных
  - Python программирование

Развертывание ML-моделей требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, 

# Итерация 2

Добавление новых концептов в онтологию


In [17]:
with onto:
  class Специалист(Thing):
        pass

  class Должность(Thing):
        pass

  class обладает_знанием(ObjectProperty):
        domain = [Специалист]
        range = [Знание]

  class обладает_компетенцией(ObjectProperty):
        domain = [Специалист]
        range = [Компетенция]

  class может_занимать_должность(ObjectProperty):
        domain = [Специалист]
        range = [Должность]

  class требует_компетенцию(ObjectProperty):
        domain = [Должность]
        range = [Компетенция]

  class имя_сотрудника(DataProperty):
        domain = [Специалист]
        range = [str]

In [18]:
employees_df.groupby(['employee_id', 'full_name'])['knowledge_id'].apply(list)

employee_id  full_name                     
1            Иванов Алексей Сергеевич           [1, 4, 5]
2            Петрова Мария Владимировна         [1, 4, 5]
3            Сидоров Дмитрий Петрович           [1, 2, 9]
4            Козлова Анна Игоревна              [5, 6, 7]
5            Федоров Максим Андреевич           [5, 6, 7]
6            Николаева Екатерина Дмитриевна    [4, 5, 10]
7            Орлов Сергей Викторович            [1, 2, 3]
Name: knowledge_id, dtype: object

### Создание индивидуальностей для сотрудников и их отношений с знаниями


In [19]:
def create_employees(employees_df, knowledge_instances):
  employees_df = employees_df.drop_duplicates()
  employee_knowledge = employees_df.groupby(['employee_id', 'full_name'])['knowledge_id'] \
.apply(lambda x: list(set(x))).reset_index() # apply(lambda x: list(set(x))) - для каждой группы собираем уникальные знания в список reset_index() - превращаем группировку обратно в DataFrame

  employee_instances = {}
  for _, row in employee_knowledge.iterrows(): # Проходим по каждой строке сгруппированных данных
    # создадим сотрудника
      employee = Специалист(f"сотрудник_{row['employee_id']}")# Создаем экземпляр Специалист с уникальным именем
      employee.имя_сотрудника = [row['full_name']]# Добавляем имя сотрудника
      employee_instances[row['employee_id']] = employee # Сохраняем в словарь

      # добавим знания сотруднику
      for knowledge_id in row['knowledge_id']:
          if knowledge_id in knowledge_instances:
              employee.обладает_знанием.append(knowledge_instances[knowledge_id])

  return employee_instances

In [20]:
employees = create_employees(employees_df, knowledges)

In [21]:
def print_employee_mindmap():
  print("\n" + "="*60)
  print("КАРТА ЗНАНИЙ КАЖДОГО СОТРУДНИКА")
  print("="*60)
  for employee_id, employee in employees.items():
          employee_name = employee.имя_сотрудника[0] if employee.имя_сотрудника else f"Сотрудник {employee_id}"

          print(f"\n{employee_name}:")

          # Получаем список знаний сотрудника
          knowledge_list = list(employee.обладает_знанием)

          if knowledge_list:
              print("Знания:")
              for knowledge in knowledge_list:
                  know_name = knowledge.название[0] if knowledge.название else knowledge.name
                  print(f"  - {know_name}")
          else:
              print("Знания: нет")

print_employee_mindmap()


КАРТА ЗНАНИЙ КАЖДОГО СОТРУДНИКА

Иванов Алексей Сергеевич:
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Петрова Мария Владимировна:
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Сидоров Дмитрий Петрович:
Знания:
  - Математическая статистика
  - Линейная алгебра
  - SQL и базы данных

Козлова Анна Игоревна:
Знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Федоров Максим Андреевич:
Знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Николаева Екатерина Дмитриевна:
Знания:
  - Визуализация данных
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Орлов Сергей Викторович:
Знания:
  - Математическая статистика
  - Линейная алгебра
  - Теория вероятностей


### Создание индивидуальностей для должностей и из связей с компетенциями


In [22]:
# Создаем экземпляры должностей
def create_position_instances(positions_df):
    position_instances = {}
    for _, row in positions_df.iterrows():
        position = Должность(f"должность_{row['position_id']}") # Создаем экземпляр Должность
        position.name = row['position_name']
        position_instances[row['position_id']] = position
    return position_instances

In [23]:
positions = create_position_instances(positions_df)
positions

{1: ml_specialist_v1.ML Engineer,
 2: ml_specialist_v1.Data Scientist,
 3: ml_specialist_v1.Data Analyst,
 4: ml_specialist_v1.NLP Engineer,
 5: ml_specialist_v1.Computer Vision Engineer}

In [24]:
# Устанавливаем связи между должностями и компетенциями
def set_position_competency_relations(position_competency_df, position_instances, competency_instances):
    for _, row in position_competency_df.iterrows():
        position = position_instances[row['position_id']]
        competency = competency_instances[row['competency_id']]
        position.требует_компетенцию.append(competency)

In [25]:
set_position_competency_relations(position_competency_df, positions, competencies)

## Выведем требования к должностям

In [26]:
# Проверим требования должностей
def print_position_require():
  print("\n" + "="*60)
  print("ТРЕБОВАНИЯ ДОЛЖНОСТЕЙ")
  print("="*60)

  for position_id, position in positions.items():
      pos_name = position.name
      print(f"\n{pos_name} требует компетенции:")

      required_competencies = list(position.требует_компетенцию)
      if required_competencies:
          for comp in required_competencies:
              print(f"  - {comp.название[0]}")
      else:
          print("  - нет требований")

print_position_require()


ТРЕБОВАНИЯ ДОЛЖНОСТЕЙ

ML Engineer требует компетенции:
  - Программирование и алгоритмы
  - Машинное обучение

Data Scientist требует компетенции:
  - Программирование и алгоритмы
  - Машинное обучение

Data Analyst требует компетенции:
  - Обработка и анализ данных

NLP Engineer требует компетенции:
  - Программирование и алгоритмы

Computer Vision Engineer требует компетенции:
  - Машинное обучение


## Создание аксиом для онтологии

 <b>Аксиома 1:</b> Если специалист обладает всеми требуемыми знаниями для компетенции, то у него есть эта компетенция

 <b>Аксиома 2:</b> Если у специалиста есть требуемые компетенции для должности, то он может занимать эту должность

  <b> Аксиома 3:</b> (для проверки корректности онтологии) Если специалист занимает должность, то у него должны быть соответствующие компетенции
      


In [27]:
def define_rules():
    # Аксиома 1:
    with onto:
                # Создаем класс-правило, наследующий от Специалист >> bool
                # >> bool означает "функция, принимающая Специалист и возвращающая bool"
        class ИмеетКомпетенциюПоЗнаниям(Специалист >> bool):
            def __init__(self): 
                super().__init__() # Вызов конструктора родительского класса

            def __call__(self, specialist):
                # Для каждой компетенции проверяем, есть ли у специалиста все необходимые знания
                for competency in onto.Компетенция.instances(): # instances() - возвращает все экземпляры класса Компетенция
                    required_knowledge = list(competency.требует_знание)  # Получаем список знаний, необходимых для компетенции
                    specialist_knowledge = list(specialist.обладает_знанием) # Получаем список знаний сотрудника

                    # Проверяем, что все требуемые знания есть у специалиста
                    if all(knowledge in specialist_knowledge for knowledge in required_knowledge):
                        if competency not in specialist.обладает_компетенцией:
                            specialist.обладает_компетенцией.append(competency)

                return True

        # Аксиома 2:
        class МожетЗаниматьДолжностьПоКомпетенциям(Специалист >> bool):
            def __init__(self):
                super().__init__()

            def __call__(self, specialist):
                # Для каждой должности проверяем, есть ли у специалиста все необходимые компетенции
                for position in onto.Должность.instances():
                    required_competencies = list(position.требует_компетенцию)
                    specialist_competencies = list(specialist.обладает_компетенцией)

                    # Проверяем, что все требуемые компетенции есть у специалиста
                    if all(competency in specialist_competencies for competency in required_competencies):
                        if position not in specialist.может_занимать_должность:
                            specialist.может_занимать_должность.append(position)

                return True

        # Аксиома 3:
        class ПроверитьКомпетенцииДляДолжности(Специалист >> bool):
            def __init__(self):
                super().__init__()

            def __call__(self, specialist):
                positions = list(specialist.может_занимать_должность)
                competencies = list(specialist.обладает_компетенцией)

                for position in positions:
                    required_competencies = list(position.требует_компетенцию)
                    if not all(comp in competencies for comp in required_competencies):
                        print(f"Предупреждение: у {specialist.name} нет всех компетенций для {position.name}")

                return True

    return ИмеетКомпетенциюПоЗнаниям(), МожетЗаниматьДолжностьПоКомпетенциям(), ПроверитьКомпетенцииДляДолжности()

In [28]:
rule1, rule2, rule3 = define_rules()

In [29]:
# Проверим связи знаний и компетенций
print_competency_knowledge()


СВЯЗИ ЗНАНИЙ И КОМПЕТЕНЦИЙ

Математическая подготовка требует знания:
  - Математическая статистика
  - Линейная алгебра
  - Теория вероятностей

Программирование и алгоритмы требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Машинное обучение требует знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Глубокое обучение требует знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Обработка и анализ данных требует знания:
  - Математическая статистика
  - SQL и базы данных

Работа с большими данными требует знания:
  - Работа с большими данными (Spark, Hadoop)
  - SQL и базы данных

Визуализация и представление данных требует знания:
  - Визуализация данных
  - Python программирование

Развертывание ML-моделей требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, 

In [30]:
onto.Специалист.instances() # instances() - возвращает все экземпляры класса

[ml_specialist_v1.сотрудник_1,
 ml_specialist_v1.сотрудник_2,
 ml_specialist_v1.сотрудник_3,
 ml_specialist_v1.сотрудник_4,
 ml_specialist_v1.сотрудник_5,
 ml_specialist_v1.сотрудник_6,
 ml_specialist_v1.сотрудник_7]

In [31]:
for employee in onto.Специалист.instances():
        rule1(employee)  # Применяем правило для компетенций
        rule2(employee)  # Применяем правило для должностей
        rule3(employee)  # Проверяем соответствие

## Проведение анализа данных

In [32]:
def print_position_for_employee():
  print("\n" + "="*60)
  print("АНАЛИЗ ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ У СОТРУДНИКОВ")
  print("="*60)

  for employee in onto.Специалист.instances():
          print(f"\nСотрудник: {employee.имя_сотрудника[0]}")

          print("Знания:")
          for knowledge in employee.обладает_знанием:
              print(f"  - {knowledge.название[0]}")

          print("Компетенции:")
          for competency in employee.обладает_компетенцией:
              print(f"  - {competency.название[0]}")

          print("Может занимать должности:")
          for position in employee.может_занимать_должность:
              print(f"  - {position.name}")

          print("-" * 30)

print_position_for_employee()


АНАЛИЗ ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ У СОТРУДНИКОВ

Сотрудник: Иванов Алексей Сергеевич
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
Компетенции:
  - Программирование и алгоритмы
  - Машинное обучение
  - Развертывание ML-моделей
Может занимать должности:
  - ML Engineer
  - Data Scientist
  - NLP Engineer
  - Computer Vision Engineer
------------------------------

Сотрудник: Петрова Мария Владимировна
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
Компетенции:
  - Программирование и алгоритмы
  - Машинное обучение
  - Развертывание ML-моделей
Может занимать должности:
  - ML Engineer
  - Data Scientist
  - NLP Engineer
  - Computer Vision Engineer
------------------------------

Сотрудник: Сидоров Дмитрий Петрович
Знания:
  - Математическая статистика
  - Линейная алгебра
  - SQL и базы данных
Компетенции:
  - Обработка и анализ данных
Может зани

In [33]:
# Убедимся, что у сотрудников достаточно знаний для получения компетенций
def check_employee_competencies(employee_instances, competency_instances):
    """Проверяем, какие компетенции могут получить сотрудники"""
    print("\n" + "="*60)
    print("ПРОВЕРКА ВОЗМОЖНЫХ КОМПЕТЕНЦИЙ СОТРУДНИКОВ")
    print("="*60)

    for employee_id, employee in employee_instances.items():
        employee_name = employee.имя_сотрудника[0]
        print(f"\n{employee_name}:")

        for competency_id, competency in competency_instances.items():
            comp_name = competency.название[0]
            required_knowledge = set(competency.требует_знание)
            employee_knowledge = set(employee.обладает_знанием)

            # Проверяем, есть ли у сотрудника все необходимые знания
            if required_knowledge.issubset(employee_knowledge):
                print(f"  Может получить компетенцию: {comp_name}")
                # Присваиваем компетенцию
                if competency not in employee.обладает_компетенцией:
                    employee.обладает_компетенцией.append(competency)
            else:
                missing_knowledge = required_knowledge - employee_knowledge
                if missing_knowledge:
                    print(f"  Не хватает для '{comp_name}':")
                    for know in missing_knowledge:
                        print(f"      - {know.название[0]}")


In [34]:
check_employee_competencies(employees, competencies)


ПРОВЕРКА ВОЗМОЖНЫХ КОМПЕТЕНЦИЙ СОТРУДНИКОВ

Иванов Алексей Сергеевич:
  Не хватает для 'Математическая подготовка':
      - Теория вероятностей
      - Линейная алгебра
  Может получить компетенцию: Программирование и алгоритмы
  Может получить компетенцию: Машинное обучение
  Не хватает для 'Глубокое обучение':
      - Компьютерное зрение
      - Обработка естественного языка (NLP)
  Не хватает для 'Обработка и анализ данных':
      - SQL и базы данных
  Не хватает для 'Работа с большими данными':
      - SQL и базы данных
      - Работа с большими данными (Spark, Hadoop)
  Не хватает для 'Визуализация и представление данных':
      - Визуализация данных
  Может получить компетенцию: Развертывание ML-моделей

Петрова Мария Владимировна:
  Не хватает для 'Математическая подготовка':
      - Теория вероятностей
      - Линейная алгебра
  Может получить компетенцию: Программирование и алгоритмы
  Может получить компетенцию: Машинное обучение
  Не хватает для 'Глубокое обучение':
      -

In [35]:
def check_employee_positions(employee_instances, position_instances):
    """Проверяем, какие должности могут занять сотрудники"""
    print("\n" + "="*60)
    print("ПРОВЕРКА ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ СОТРУДНИКОВ")
    print("="*60)

    for employee_id, employee in employee_instances.items():
        employee_name = employee.имя_сотрудника[0]
        print(f"\n{employee_name}:")

        for position_id, position in position_instances.items():
            pos_name = position.name
            required_competencies = set(position.требует_компетенцию)
            employee_competencies = set(employee.обладает_компетенцией)

            # Проверяем, есть ли у сотрудника все необходимые компетенции
            if required_competencies.issubset(employee_competencies): #Метод issubset() - проверяет, является ли одно множество подмножеством другого.
                print(f"   Может занять должность: {pos_name}")
                # Присваиваем должность
                if position not in employee.может_занимать_должность:
                    employee.может_занимать_должность.append(position)
            else:
                missing_competencies = required_competencies - employee_competencies
                if missing_competencies:
                    print(f"   Не хватает для '{pos_name}':")
                    for comp in missing_competencies:
                        print(f"      - {comp.название[0]}")

# Применяем проверки

check_employee_positions(employees, positions)



ПРОВЕРКА ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ СОТРУДНИКОВ

Иванов Алексей Сергеевич:
   Может занять должность: ML Engineer
   Может занять должность: Data Scientist
   Не хватает для 'Data Analyst':
      - Обработка и анализ данных
   Может занять должность: NLP Engineer
   Может занять должность: Computer Vision Engineer

Петрова Мария Владимировна:
   Может занять должность: ML Engineer
   Может занять должность: Data Scientist
   Не хватает для 'Data Analyst':
      - Обработка и анализ данных
   Может занять должность: NLP Engineer
   Может занять должность: Computer Vision Engineer

Сидоров Дмитрий Петрович:
   Не хватает для 'ML Engineer':
      - Машинное обучение
      - Программирование и алгоритмы
   Не хватает для 'Data Scientist':
      - Машинное обучение
      - Программирование и алгоритмы
   Может занять должность: Data Analyst
   Не хватает для 'NLP Engineer':
      - Программирование и алгоритмы
   Не хватает для 'Computer Vision Engineer':
      - Машинное обучение

Козлова Анна 

# Задание 1:  Добавление необходимой компетенции в должность

Добавить требование, что компетениция NLP Enginner содержит знание по глубокому обучению.

<B>Важно, что добавление в онтологию новых данных в Питоне должно сопровождаться созданием новой онтологии.</B>


In [36]:
positions

{1: ml_specialist_v1.ML Engineer,
 2: ml_specialist_v1.Data Scientist,
 3: ml_specialist_v1.Data Analyst,
 4: ml_specialist_v1.NLP Engineer,
 5: ml_specialist_v1.Computer Vision Engineer}

In [37]:
positions[4].требует_компетенцию

[ml_specialist_v1.компетенция_2]

In [38]:
competencies[2].название

['Программирование и алгоритмы']

In [39]:
for i, item in competencies.items():
  print (i, ' ', item.название)


1   ['Математическая подготовка']
2   ['Программирование и алгоритмы']
3   ['Машинное обучение']
4   ['Глубокое обучение']
5   ['Обработка и анализ данных']
6   ['Работа с большими данными']
7   ['Визуализация и представление данных']
8   ['Развертывание ML-моделей']


In [40]:
positions[4].требует_компетенцию.append(competencies[4])

In [41]:
with onto:

    # Получаем должность NLP Engineer (индекс 4)
    # positions — словарь, где ключ — числовой ID должности, значение — объект-индивид класса Должность.
    # Индекс 4 соответствует должности "NLP Engineer" 
    target_position = positions[4]

    # Получаем компетенцию "Глубокое обучение" (индекс 4)
    # competencies — словарь, где ключ — ID компетенции, значение — индивид класса Компетенция.
    # Индекс 4 — это компетенция "Глубокое обучение" 
    target_competency = competencies[4]

    # Проверяем, не добавлена ли уже эта компетенция в список требований должности.
    # Свойство 'требует_компетенцию' — это ObjectProperty, ведущее себя как список Python.
    # Проверка 'not in' предотвращает дублирование связей.
    if target_competency not in target_position.требует_компетенцию:

        # Добавляем компетенцию в список требований должности.
        # .append() добавляет объект target_competency в конец списка.
        target_position.требует_компетенцию.append(target_competency)

        # Выводим информационное сообщение об успешном добавлении.
        # target_position.name — имя индивида (например, "NLP Engineer").
        # target_competency.название[0] — первое значение дата-свойства 'название' (строка).
        print(f"Добавлено: должность '{target_position.name}' требует компетенцию '{target_competency.название[0]}'")

    else:
        # Если компетенция уже присутствует, выводим сообщение об этом.
        print(f"Компетенция '{target_competency.название[0]}' уже есть в требованиях")


print("\nТекущие требования к должности NLP Engineer:")

# Перебираем все компетенции, которые теперь требуются для должности NLP Engineer.
# positions[4] — снова получаем объект должности, свойство 'требует_компетенцию' — список компетенций.
for comp in positions[4].требует_компетенцию:

    # Для каждой компетенции получаем её название для вывода.
    # Если у компетенции есть свойство 'название' и оно не пустое, берём первый элемент.
    # Иначе используем внутреннее имя индивида (comp.name, например "компетенция_4").
    comp_name = comp.название[0] if comp.название else comp.name

    # Выводим название компетенции с дефисом (маркер списка).
    print(f" - {comp_name}")

Компетенция 'Глубокое обучение' уже есть в требованиях

Текущие требования к должности NLP Engineer:
 - Программирование и алгоритмы
 - Глубокое обучение


###

## Задание 2:
Добавьте требование в онтологию: должность Data Scientist требует компетенции "Математическая подготовка"

In [42]:
with onto:

    # Получаем должность Data Scientist (индекс 2)
    # positions — словарь, созданный ранее, где ключ — числовой ID должности,
    # значение — объект-индивид класса Должность.
    # Индекс 2 соответствует должности "Data Scientist" 
    ds_position = positions[2]

    # Получаем компетенцию "Математическая подготовка" (индекс 1)
    # competencies — словарь, где ключ — ID компетенции, значение — индивид класса Компетенция.
    # Индекс 1 — это компетенция "Математическая подготовка" 
    math_competency = competencies[1]

    # Добавляем новое требование
    # Проверяем, не добавлена ли уже эта компетенция в список требований должности.
    # Свойство 'требует_компетенцию' — это ObjectProperty, ведущее себя как список Python.
    if math_competency not in ds_position.требует_компетенцию:

        # Добавляем компетенцию в список требований должности.
        # .append() добавляет объект math_competency в конец списка.
        ds_position.требует_компетенцию.append(math_competency)

        # Выводим информационное сообщение об успешном добавлении.
        # math_competency.название[0] — первое значение дата-свойства 'название' (строка "Математическая подготовка").
        # ds_position.name — имя индивида (например, "Data Scientist").
        print(f"Компетенция '{math_competency.название[0]}' добавлена к должности '{ds_position.name}'")


print("\nОбновленный список требований для Data Scientist:")

# Перебираем все компетенции, которые теперь требуются для должности Data Scientist.
# positions[2] — снова получаем объект должности, свойство 'требует_компетенцию' — список компетенций.
for comp in positions[2].требует_компетенцию:

    # Для каждой компетенции получаем её название для вывода.
    # Если у компетенции есть свойство 'название' и оно не пустое, берём первый элемент.
    # Иначе используем внутреннее имя индивида (comp.name, например "компетенция_1").
    comp_name = comp.название[0] if comp.название else comp.name

    # Выводим название компетенции с дефисом (маркер списка).
    print(f" - {comp_name}")

Компетенция 'Математическая подготовка' добавлена к должности 'Data Scientist'

Обновленный список требований для Data Scientist:
 - Программирование и алгоритмы
 - Машинное обучение
 - Математическая подготовка


## Задание 3:
Добавьте сотруднику Федорову Максиму Андреевичу необходимые знания, чтобы он мог претендовать на должность Computer Vision Engineer

In [43]:
# Находим сотрудника по имени
# Инициализируем переменную target_employee значением None.
# Если сотрудник будет найден, в неё запишется объект индивида.
target_employee = None

# Перебираем все пары (ключ, значение) в словаре employees.
# emp_id — числовой ID сотрудника (1,2,3...), emp — объект-индивид класса Специалист.
for emp_id, emp in employees.items():

    # Проверяем, есть ли у сотрудника свойство 'имя_сотрудника' (DataProperty)
    # и содержит ли его значение строку "Федоров Максим Андреевич".
    # Свойство 'имя_сотрудника' хранит список строк, поэтому обращаемся к первому элементу [0].
    if emp.имя_сотрудника and "Федоров Максим Андреевич" in emp.имя_сотрудника[0]:

        # Если нашли нужного сотрудника, записываем его в переменную target_employee.
        target_employee = emp

        # Прерываем цикл, так как дальнейший перебор не нужен.
        break

# Проверяем, был ли найден сотрудник (target_employee не равен None).
if target_employee:

    # Выводим имя найденного сотрудника.
    # target_employee.имя_сотрудника[0] — первое (и единственное) значение свойства 'имя_сотрудника'.
    print(f"\nСотрудник: {target_employee.имя_сотрудника[0]}")

    # Определяем целевую должность
    # positions[5] — из словаря positions получаем должность с ID=5.
    # В исходных данных это "Computer Vision Engineer".
    cv_position = positions[5]  # Computer Vision Engineer

    # Собираем все знания, необходимые для должности
    # Создаём пустое множество для хранения уникальных объектов Знание.
    required_knowledge_set = set()

    # Перебираем все компетенции, которые требуются для должности cv_position.
    # cv_position.требует_компетенцию — список объектов Компетенция.
    for competency in cv_position.требует_компетенцию:

        # Для каждой компетенции перебираем все знания, которые она требует.
        # competency.требует_знание — список объектов Знание.
        for knowledge in competency.требует_знание:

            # Добавляем знание в множество required_knowledge_set.
            # Множество автоматически исключает дубликаты.
            required_knowledge_set.add(knowledge)

    # Выводим название целевой должности.
    # cv_position.name — внутреннее имя индивида (например, "Computer Vision Engineer").
    print(f"\nЦелевая должность: {cv_position.name}")

    # Выводим заголовок списка необходимых знаний.
    print(f"Необходимые знания для должности:")

    # Перебираем каждое знание из множества required_knowledge_set.
    for k in required_knowledge_set:

        # Получаем название знания для вывода.
        # Если у знания есть свойство 'название' (DataProperty) и оно не пустое,
        # берём первый элемент списка (k.название[0]).
        # Иначе используем внутреннее имя индивида (k.name, например "знание_1").
        k_name = k.название[0] if k.название else k.name

        # Выводим название знания с дефисом (маркер списка).
        print(f"  - {k_name}")


Сотрудник: Федоров Максим Андреевич

Целевая должность: Computer Vision Engineer
Необходимые знания для должности:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)


In [44]:
with onto:

    # Добавляем недостающие знания

    # Преобразуем текущий список знаний сотрудника в множество.
    # target_employee.обладает_знанием — это список объектов Знание (ObjectProperty).
    # Множество нужно для быстрого выполнения операций над множествами (разность, пересечение).
    current_knowledge = set(target_employee.обладает_знанием)

    # Вычисляем недостающие знания: из множества требуемых знаний (required_knowledge_set)
    # вычитаем множество уже имеющихся знаний.
    # Результат — множество знаний, которых у сотрудника пока нет.
    missing_knowledge = required_knowledge_set - current_knowledge

    # Проверяем, есть ли недостающие знания (множество не пустое).
    if missing_knowledge:

        # Выводим информационное сообщение о начале добавления.
        print(f"\nДобавляем недостающие знания:")

        # Перебираем каждое знание из множества missing_knowledge.
        for knowledge in missing_knowledge:

            # Добавляем знание в свойство 'обладает_знанием' сотрудника.
            # Свойство является ObjectProperty и хранит список знаний.
            # .append() добавляет объект knowledge в конец списка.
            target_employee.обладает_знанием.append(knowledge)

            # Получаем название знания для вывода в консоль.
            # Если у знания есть свойство 'название' (DataProperty) и оно не пустое,
            # берём первый элемент списка (knowledge.название[0]).
            # Иначе используем внутреннее имя индивида (knowledge.name).
            k_name = knowledge.название[0] if knowledge.название else knowledge.name

            # Выводим название добавленного знания с плюсиком (символизирует добавление).
            print(f"  + {k_name}")

    else:
        # Если недостающих знаний нет (множество пустое), сообщаем об этом.
        print(f"\nУ сотрудника уже есть все необходимые знания")

    # Выводим заголовок для обновлённого списка знаний сотрудника.
    print(f"\nОбновленный список знаний сотрудника:")

    # Перебираем все знания, которыми теперь обладает сотрудник (после добавлений).
    for k in target_employee.обладает_знанием:

        # Аналогично получаем название знания для вывода.
        k_name = k.название[0] if k.название else k.name

        # Выводим название знания с дефисом (маркер списка).
        print(f"  - {k_name}")



Добавляем недостающие знания:
  + Математическая статистика
  + Python программирование

Обновленный список знаний сотрудника:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение
  - Математическая статистика
  - Python программирование


In [45]:
# Перебираем всех специалистов (сотрудников) в онтологии.
# onto.Специалист.instances() возвращает список всех индивидов класса Специалист,
# которые были созданы в онтологии (включая исходных сотрудников и добавленных позже).
for emp in onto.Специалист.instances():

    # Применяем к сотруднику правило №1 (аксиома 1):
    # "Если специалист обладает всеми требуемыми знаниями для компетенции, то он получает эту компетенцию".
    # rule1(emp) – это функция, которая проверяет знания сотрудника и
    # автоматически добавляет ему соответствующие компетенции в свойство 'обладает_компетенцией'.
    rule1(emp)

    # Применяем к сотруднику правило №2 (аксиома 2):
    # "Если у специалиста есть все требуемые компетенции для должности, то он может занимать эту должность".
    # rule2(emp) – функция, которая на основе компетенций сотрудника
    # добавляет ему подходящие должности в свойство 'может_занимать_должность'.
    rule2(emp)

In [46]:
def print_position_for_employee():

    # Перебираем всех специалистов (сотрудников) в онтологии.
    # onto.Специалист.instances() возвращает список всех индивидов класса Специалист.
    for employee in onto.Специалист.instances():

        # Выводим имя сотрудника.
        # employee.имя_сотрудника — это DataProperty, хранящее список строк.
        # Берём первый элемент списка [0], так как имя обычно одно.
        print(f"\nСотрудник: {employee.имя_сотрудника[0]}")

        # Выводим заголовок списка должностей.
        print("Может занимать должности:")

        # Перебираем все должности, которые доступны сотруднику.
        # employee.может_занимать_должность — это ObjectProperty, ведущее себя как список.
        # Оно заполняется правилом rule2(employee) на основе компетенций сотрудника.
        for position in employee.может_занимать_должность:

            # Выводим название должности.
            # position.name — внутреннее имя индивида (например, "ML Engineer").
            print(f"  - {position.name}")

        # Выводим разделитель (30 дефисов) для визуального отделения разных сотрудников.
        print("-" * 30)

# Вызываем функцию, чтобы увидеть результат в консоли.
print_position_for_employee()


Сотрудник: Иванов Алексей Сергеевич
Может занимать должности:
  - ML Engineer
  - Data Scientist
  - NLP Engineer
  - Computer Vision Engineer
------------------------------

Сотрудник: Петрова Мария Владимировна
Может занимать должности:
  - ML Engineer
  - Data Scientist
  - NLP Engineer
  - Computer Vision Engineer
------------------------------

Сотрудник: Сидоров Дмитрий Петрович
Может занимать должности:
  - Data Analyst
------------------------------

Сотрудник: Козлова Анна Игоревна
Может занимать должности:
------------------------------

Сотрудник: Федоров Максим Андреевич
Может занимать должности:
  - ML Engineer
  - NLP Engineer
  - Computer Vision Engineer
------------------------------

Сотрудник: Николаева Екатерина Дмитриевна
Может занимать должности:
  - NLP Engineer
------------------------------

Сотрудник: Орлов Сергей Викторович
Может занимать должности:
------------------------------


## Задание 4:
Добавьте требование в онтологию нового сотрудника, укажите в качестве имя_сотрудника ваше ФИО. Добавьте знания для данного сотрудника, чтобы он смог претендовать на все должности текущей онтологии

In [47]:
with onto:

    # Определяем переменную your_name со строковым значением – ФИО нового сотрудника.
    your_name = "Корчагина Александра Игоревна"

    # Вычисляем новый идентификатор сотрудника: берём максимальный ключ из словаря employees
    # и прибавляем 1. Это гарантирует уникальность ID.
    new_employee_id = max(employees.keys()) + 1   # следующий доступный id

    # Создаём индивид (экземпляр) класса Специалист с уникальным именем вида "сотрудник_<id>".
    # Имя должно быть уникальным в пределах онтологии.
    new_employee = Специалист(f"сотрудник_{new_employee_id}")

    # Заполняем дата-свойство 'имя_сотрудника' – список строк (обычно один элемент).
    # Свойство определено как DataProperty с domain = Специалист, range = str.
    new_employee.имя_сотрудника = [your_name]

    # Добавляем созданного сотрудника в словарь employees под ключом new_employee_id,
    # чтобы в дальнейшем иметь быстрый доступ к нему.
    employees[new_employee_id] = new_employee

    # Выводим сообщение об успешном создании сотрудника с его ID и ФИО.
    print(f"Создана сотрудница: {your_name} (id={new_employee_id})")

    # Преобразуем значения словаря knowledges (все объекты Знание) в список.
    all_knowledge = list(knowledges.values())

    # Перебираем каждое знание в списке.
    for kn in all_knowledge:

        # Проверяем, есть ли уже это знание у сотрудника (чтобы избежать дублирования).
        # Свойство 'обладает_знанием' – это список (ObjectProperty), поэтому можно использовать in.
        if kn not in new_employee.обладает_знанием:

            # Добавляем знание в список знаний сотрудника.
            new_employee.обладает_знанием.append(kn)

    # Выводим количество добавленных знаний (всего существующих в онтологии).
    print(f"Добавлено {len(all_knowledge)} знаний")

    # rule1 – аксиома: если сотрудник обладает всеми знаниями, требуемыми для компетенции,
    # то он получает эту компетенцию (добавляется в свойство 'обладает_компетенцией').
    rule1(new_employee)

    # rule2 – аксиома: если сотрудник обладает всеми компетенциями, требуемыми для должности,
    # то он может занимать эту должность (добавляется в свойство 'может_занимать_должность').
    rule2(new_employee)

    # Выводим заголовок.
    print(f"\n{your_name} может занимать следующие должности:")

    # Перебираем все должности, которые были добавлены в свойство 'может_занимать_должность'
    # после применения правил (т.е. те, для которых есть все требуемые компетенции).
    for pos in new_employee.может_занимать_должность:

        # Выводим название должности. У индивида класса Должность есть свойство 'name' (задаётся при создании).
        print(f"  - {pos.name}")

    # Дополнительная проверка: убедимся, что доступны все 5 должностей из онтологии.

    # Создаём множество ожидаемых должностей – из всех значений словаря positions.
    # positions.values() возвращает все объекты Должность, имеющиеся в онтологии.
    expected_positions = set(positions.values())

    # Создаём множество должностей, которые фактически доступны сотруднику
    # (преобразуем список 'может_занимать_должность' в множество для сравнения).
    available = set(new_employee.может_занимать_должность)

    # Проверяем, является ли множество ожидаемых должностей подмножеством доступных.
    # Если все ожидаемые должности есть в available, значит сотрудник может претендовать на все.
    if expected_positions.issubset(available):

        # Выводим сообщение об успехе – все должности доступны.
        print("\n Сотрудница может претендовать на ВСЕ должности!")

    else:
        # Если не все – вычисляем, каких именно должностей не хватает (разность множеств).
        missing = expected_positions - available

        # Выводим список названий недостающих должностей.
        print(f"\n Не может занять: {[p.name for p in missing]}")

Создана сотрудница: Корчагина Александра Игоревна (id=8)
Добавлено 10 знаний

Корчагина Александра Игоревна может занимать следующие должности:
  - ML Engineer
  - Data Scientist
  - Data Analyst
  - NLP Engineer
  - Computer Vision Engineer

 Сотрудница может претендовать на ВСЕ должности!
